In [1]:
import pandas as pd
import json
import geopandas as gpd
from shapely.geometry import LineString, Point
from pathlib import Path

pd.set_option('display.max_columns', None)

In [2]:
# 1. Đọc file JSON thô và tách ways, nodes có sẵn
file_path = r'C:\Tailieu\PPNC\data\data\raw\thanh_my_tay_full_20261110_141005.json'

with open(file_path, 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

elements = raw_data.get('elements', [])
ways = [el for el in elements if el.get('type') == 'way']
nodes = [el for el in elements if el.get('type') == 'node']

print('Số điểm tọa độ (nodes):', len(nodes))
print('Số đoạn đường/hẻm (ways):', len(ways))

Số điểm tọa độ (nodes): 6845
Số đoạn đường/hẻm (ways): 1675


In [3]:
# 2. Tạo bảng dữ liệu các điểm tọa độ có sẵn (df_nodes)
df_nodes = pd.DataFrame(nodes)

# Tạo dict {node_id: (lon, lat)} để ghép tọa độ vào ways
node_dict = dict(zip(df_nodes['id'], zip(df_nodes['lon'], df_nodes['lat'])))

df_nodes.head()

,type,id,lat,lon
0,node,366368194,10.790241,106.709544
1,node,366376424,10.789385,106.710677
2,node,366384879,10.789365,106.708529
3,node,366386086,10.790292,106.708695
4,node,366388178,10.789719,106.708615


In [4]:
# 3. Đưa toàn bộ dữ liệu có sẵn của ways vào GeoDataFrame (gdf)
data = []

for el in ways:
    node_ids = [nid for nid in el.get('nodes', []) if nid in node_dict]
    if len(node_ids) >= 2:
        coords = [node_dict[nid] for nid in node_ids]
        row = {
            'id': el['id'],
            'type': el['type'],
            'nodes': node_ids,
            **el.get('tags', {}),
            'geometry': LineString(coords)
        }
        data.append(row)

gdf = gpd.GeoDataFrame(data, geometry='geometry', crs='EPSG:4326')

print('Số lượng đối tượng:', len(gdf))
print('Tổng số cột ban đầu:', len(gdf.columns))
print('Các cột dữ liệu:', gdf.columns.tolist())

Số lượng đối tượng: 1675
Tổng số cột ban đầu: 83
Các cột dữ liệu: ['id', 'type', 'nodes', 'highway', 'maxweight:forward', 'maxweightrating:hgv:forward', 'name', 'sidewalk', 'geometry', 'lane_markings', 'name:vi', 'surface', 'bicycle', 'cycleway', 'foot', 'lanes', 'maxspeed', 'motor_vehicle', 'name:en', 'old_name', 'oneway', 'oneway:bicycle', 'oneway:motorcycle', 'bridge', 'bridge:name', 'layer', 'lit', 'maxspeed:bus', 'maxspeed:hgv', 'maxspeed:motorcar', 'motorcycle', 'name:ko', 'name:th', 'alt_name', 'alt_name:en', 'motorcar:conditional', 'horse', 'maxweight:hgv', 'maxweight:hgv:conditional', 'maxweight:hgv_articulated', 'bus', 'maxheight', 'motorcar', 'name:zh', 'maxweight', 'sidewalk:left', 'sidewalk:right', 'sidewalk:right:surface', 'motorroad', 'service', 'width', 'addr:city', 'addr:district', 'cutting', 'junction', 'access', 'destination', 'tunnel', 'source:maxspeed', 'handrail', 'ramp', 'ramp:bicycle', 'step_count', 'start_date', 'maxweightrating:hgv', 'turn:lanes', 'footway', '

In [5]:
# 4. Lọc giữ lại các cột có sẵn phục vụ bài toán đường/hẻm và chuẩn hóa kiểu dữ liệu
keep_cols = [
    'id', 'name', 'name:vi', 'highway', 'service', 'oneway', 
    'lanes', 'maxspeed', 'surface', 'width', 'sidewalk', 
    'bridge', 'tunnel', 'layer', 'lit', 'junction', 'access',
    'motorcycle', 'motorcar', 'bicycle', 'foot', 'nodes', 'geometry'
]
keep_cols = [col for col in keep_cols if col in gdf.columns]
gdf = gdf[keep_cols].copy()

# Gộp cột tên tiếng Việt (name:vi) vào cột name nếu name bị trống, sau đó bỏ cột name:vi
if 'name:vi' in gdf.columns:
    gdf['name'] = gdf['name'].fillna(gdf['name:vi'])
    gdf = gdf.drop(columns=['name:vi'])

# Chuẩn hóa các cột số có sẵn từ chuỗi (str) sang số (float)
for col in ['lanes', 'maxspeed', 'width', 'layer']:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce')

# Tổng quan dữ liệu sau khi lọc cột
print('Số cột sau khi lọc:', len(gdf.columns))
print('\n--- Kiểu dữ liệu (dtypes) ---')
print(gdf.dtypes)
print('\n--- Thống kê loại đường (highway) ---')
print(gdf['highway'].value_counts(dropna=False))
print('\n--- Kiểm tra dữ liệu thiếu (Null) ---')
print(gdf.isnull().sum())
print('\nCRS:', gdf.crs)

Số cột sau khi lọc: 22

--- Kiểu dữ liệu (dtypes) ---
id               int64
name               str
highway            str
service            str
oneway             str
lanes          float64
maxspeed       float64
surface            str
width          float64
sidewalk           str
bridge             str
tunnel             str
layer          float64
lit                str
junction           str
access             str
motorcycle         str
motorcar           str
bicycle            str
foot               str
nodes           object
geometry      geometry
dtype: object

--- Thống kê loại đường (highway) ---
highway
service           641
residential       387
footway           353
primary           116
tertiary           39
primary_link       31
trunk              31
steps              27
secondary          24
secondary_link      9
pedestrian          8
trunk_link          5
path                2
living_street       1
tertiary_link       1
Name: count, dtype: int64

--- Kiểm tra dữ liệu t

In [6]:
# 5. Xem 10 dòng đầu tiên của bảng dữ liệu chuẩn hóa (gdf)
gdf.head(10)

,id,name,highway,service,oneway,lanes,maxspeed,surface,width,sidewalk,bridge,tunnel,layer,lit,junction,access,motorcycle,motorcar,bicycle,foot,nodes,geometry
0,32575790,Hẻm 31 Ung Văn Khiêm,residential,NaN,NaN,NaN,NaN,NaN,NaN,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[366414641, 5733448375, 2030638785, 2030639268...","LINESTRING (106.71347 10.80913, 106.71402 10.8..."
1,32575792,NaN,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[366452436, 5737751039, 366379167, 366404193, ...","LINESTRING (106.71882 10.81074, 106.71914 10.8..."
2,32575795,Đường D2,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[5733055246, 366467734, 5881087223, 366442186,...","LINESTRING (106.7182 10.81006, 106.71793 10.80..."
3,32575984,Mê Linh,residential,NaN,NaN,NaN,NaN,asphalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[366368194, 5671815972, 366458884, 5671815966,...","LINESTRING (106.70954 10.79024, 106.70952 10.7..."
4,32576073,Võ Oanh,tertiary,NaN,NaN,2.0,50.0,paved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,yes,"[366403574, 14302380595, 366386878, 1430269991...","LINESTRING (106.71589 10.80433, 106.71629 10.8..."
5,32576539,NaN,primary_link,NaN,NaN,NaN,NaN,asphalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[2403042655, 366381336]","LINESTRING (106.71101 10.79863, 106.71109 10.7..."
6,32577627,Huỳnh Mẫn Đạt,residential,NaN,NaN,NaN,NaN,asphalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[366386086, 5738098126, 11786621589, 577522183...","LINESTRING (106.7087 10.79029, 106.70876 10.79..."
7,32577688,Xô Viết Nghệ Tĩnh,primary,NaN,yes,2.0,NaN,asphalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[728690562, 1771088772, 366389762]","LINESTRING (106.70717 10.79245, 106.70737 10.7..."
8,32578249,Đường D5,tertiary,NaN,NaN,NaN,NaN,asphalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[10910534491, 12968646809, 2030639172]","LINESTRING (106.71189 10.80678, 106.71211 10.8..."
9,32578410,Hẻm 362 Ung Văn Khiêm,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[366421019, 4235815135, 366461809, 1497313065,...","LINESTRING (106.72361 10.80006, 106.72317 10.8..."


In [7]:
# 6. Lưu bảng ways và nodes đã chuẩn hóa cơ bản vào data/process
output_dir = Path(r'C:\Tailieu\PPNC\data\data\process')
output_dir.mkdir(parents=True, exist_ok=True)

gdf.drop(columns=['geometry']).to_csv(output_dir / 'ways.csv', index=False, encoding='utf-8-sig')
df_nodes.to_csv(output_dir / 'nodes.csv', index=False, encoding='utf-8-sig')
gdf.drop(columns=['nodes']).to_file(output_dir / 'ways.geojson', driver='GeoJSON')

print('Đã lưu ways.csv, nodes.csv và ways.geojson vào:', output_dir)

Đã lưu ways.csv, nodes.csv và ways.geojson vào: C:\Tailieu\PPNC\data\data\process
